In [ ]:
# notebook: 01_setup_verify.ipynb
import sys
import os

# 1. Danh sách các đường dẫn có thể chứa thư mục src
possible_src_paths = [
    '/home/jovyan/work/src',      # Khi chạy trên Jupyter Docker
    os.path.abspath('../src'),    # Khi chạy VS Code (thư mục hiện tại là notebooks)
    os.path.abspath('./src')      # Khi chạy VS Code (thư mục hiện tại là gốc dự án)
]

# 2. Tìm và add đường dẫn đúng vào sys.path
for path in possible_src_paths:
    if os.path.exists(path):
        if path not in sys.path:
            sys.path.insert(0, path)
        print(f"Đã load thư mục src từ: {path}")
        break
else:
    print("CẢNH BÁO: Không tìm thấy thư mục src!")

# 3. Import các cấu hình từ src
from config.spark_config import create_spark_session, bronze_path
from config.minio_config import ensure_buckets_exist

# Bước 1: Đảm bảo buckets tồn tại
ensure_buckets_exist()

# Bước 2: Tạo SparkSession (lần đầu sẽ download JARs ~200MB, mất 2-3 phút)
spark = create_spark_session('SetupVerify')

# Bước 3: Test ghi và đọc Parquet lên MinIO
test_df = spark.createDataFrame([
    (1, 'test_user_1', 'B001', 4.5),
    (2, 'test_user_2', 'B002', 3.0),
], ['id', 'user_id', 'asin', 'rating'])

test_path = bronze_path('_test/verify.parquet')

# MinIO với Spark thường cần scheme s3a:// (không phải s3://)
if test_path.startswith('s3://'):
    test_path = 's3a://' + test_path[len('s3://'):]
elif test_path.startswith('s3n://'):
    test_path = 's3a://' + test_path[len('s3n://'):]

print(f'Using test path: {test_path}')

try:
    test_df.write.mode('overwrite').parquet(test_path)
    print(f'Write OK: {test_path}')

    read_back = spark.read.parquet(test_path)
    read_back.show()
    print(f'Read OK: {read_back.count()} rows')
except Exception as e:
    print('Parquet read/write failed.')
    if hasattr(e, 'java_exception') and e.java_exception is not None:
        print('Java exception:', e.java_exception.toString())
    raise

# Bước 4: Kiểm tra Spark UI
print(f'Spark UI: http://localhost:4040')
print('Setup complete!')

Đã load thư mục src từ: /home/jovyan/work/src
Bucket exists: electronics-bronze
Bucket exists: electronics-silver
Bucket exists: electronics-gold
SparkSession created: SetupVerify
MinIO endpoint: http://http://minio:9000


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=49>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip

Py4JError: An error occurred while calling o82.parquet